# EEE 457 Transmissão de Energia Elétrica
## Escola Politécnica
## Universidade Federal do Rio de Janeiro
### Antonio C. S. Lima
Outubro 2025
### Cálculo de parâmetros unitários em linhas de transmissão aéreas 

In [16]:
# carrega as bibliotecas

import numpy as np
from numpy import pi, sqrt, log, exp, diag, eye, kron
from numpy.linalg import inv, eig
from scipy.special import iv, kv
import matplotlib.pyplot as plt
import pandas as pd
from typing import Iterable, Tuple, Optional, Union
import warnings 
import os
import line_cable_param as lcp

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "sans-serif",
    "font.size": 14,
    "font.sans-serif": "Helvetica",
})

In [17]:
# constantes importantes
µ0 = 4e-7 * pi  # permeabilidade do vácuo (H/m)
ε0 = 8.854187817e-12  # permissividade do vácuo (F/m)
c0 = 1 / sqrt(µ0 * ε0)  # velocidade da luz no vácuo (m/s)
η0 = sqrt(µ0 / ε0)  # impedância característica do vácuo (Ohms)
freq = 60  # frequência de operação (Hz)
omega = 2 * pi * freq  # frequência angular (rad/s)
a = np.exp( 2j * pi / 3)  #  1/_ 120 graus para o calculo dos parâmetros de sequência 

In [18]:
# primmeira configuração
# circuito 138 kV  138CST1
fpr = 7.8   # flecha cabo para-raios
fcd = 11.9  # flecha cabo condutor
fc = np.concat( [3 * [fcd], 2* [fpr] ] )  # m flechas dos condutores 
xc = np.array([3.4, -3.4, 3.4, -3.4, 3.4]) # coordenada x dos cabos
yc = 26.9 - np.array([2 * 3.8, 3.8+1.9, 3.8, 0, 0]) # coordenada y dos cabos
yc = yc - 2/3 * fc  # altura media dos condutores 
rho_s = 1e3                   # resistividade do solo (Ohm.m)
ri = (6.75e-3)/2              # raio interno dos condutores de fase (m)
rf = (27.01e-3)/2             # raio externo dos condutores de fase (m)
rdc = 0.0734e-3               # resistência DC dos condutores de fase (Ohm/m)
nb = 1
npr = 2
rpr = 4.57e-3                # raio dos para-raios (m) considerando 3/8" EHS
rdc_pr =  4.190e-3           # resistência DC dos para-raios (Ohm/m)

## Desprezando cabos pararraios

In [ ]:
Z, Y =lcp.czyl_simple(omega=omega,
                       x=xc[0:3], y=yc[0:3],  sigma_s=1/rho_s, rdc=rdc,
                       rf=rf, rint=ri) 

In [20]:
print("Impedance Matrix (Ohm/km):")
print(np.round(Z * 1000,5))

Impedance Matrix (Ohm/km):
[[0.13337+0.94292j 0.05877+0.45434j 0.05873+0.50109j]
 [0.05877+0.45434j 0.1333 +0.94299j 0.0587 +0.45441j]
 [0.05873+0.50109j 0.0587 +0.45441j 0.13323+0.94306j]]


In [21]:
print("\nAdmittance Matrix (S/km):")
print(Y * 1000)


Admittance Matrix (S/km):
[[3.e-09+3.07453459e-06j 0.e+00-3.89902544e-07j 0.e+00-7.02487386e-07j]
 [0.e+00-3.89902544e-07j 3.e-09+2.91396296e-06j 0.e+00-4.38334816e-07j]
 [0.e+00-7.02487386e-07j 0.e+00-4.38334816e-07j 3.e-09+2.97534194e-06j]]


In [22]:
# matriz de Fortescue
A = np.array([[1, 1, 1],[1, a**2, a],[1, a, a**2]])
A_inv = np.linalg.inv(A)
# impedancias de sequência
Z_seq = A_inv @ Z @ A
# admitancias de sequência
Y_seq = A_inv @ Y @ A

In [23]:
print(Z_seq * 1000)

[[ 0.25076705+1.88289329j -0.01346249+0.00770295j  0.01356581+0.00776241j]
 [ 0.01356581+0.00776241j  0.07456565+0.47304457j  0.02696787-0.01557003j]
 [-0.01346249+0.00770295j -0.02696798-0.01556985j  0.07456565+0.47304457j]]


In [24]:
print(Y_seq * 1000)

[[ 3.00000000e-09+1.96746333e-06j  7.25168853e-08+7.34066261e-09j
  -7.25168853e-08+7.34066261e-09j]
 [-7.25168853e-08+7.34066261e-09j  3.00000000e-09+3.49818808e-06j
  -1.98189528e-07+1.15200812e-07j]
 [ 7.25168853e-08+7.34066261e-09j  1.98189528e-07+1.15200812e-07j
   3.00000000e-09+3.49818808e-06j]]


In [25]:
z012= np.diag(Z_seq)
y012= np.diag(Y_seq)

In [26]:
y1 = y012[1] * 1000  # S/m para S/km
print(y1 * 1e6)  # µS/km

(0.003000000000000274+3.4981880762167936j)


In [27]:
z1  = z012[1] * 1000  # Ohm/m para Ohm/km
print(z1)  # Ohm/km

(0.07456565466796128+0.4730445676192674j)


In [28]:
zc = np.sqrt(z012[1]/y012[1])
print("Impedância característica (Ohm):", np.real(zc))

Impedância característica (Ohm): 368.87598719528967


In [29]:
# potencia natural do circuito
Vn = 138
Pn = Vn**2 / np.real(zc)

print(Pn)

51.627106835549476


### Incluindo os cabos pararraios 

In [32]:
Z, Y =lcp.czyl_overhead(omega=omega,
                        x=xc, y=yc,  
                        sigma_s=1/rho_s, 
                        rdc=rdc,
                        rf=rf, rint=ri,
                        npr=npr, rdcpr=rdc_pr, rpr=rpr)

In [40]:
nc = 5
npr =2
# Calculate conductor resistivities
rhoc = rdc * np.pi * (rf**2 - ri**2)
rhopr = rdc_pr * np.pi * rpr**2
    
# Internal impedance
z_internal = np.zeros(nc, dtype=complex)
if ri != 0:
        z_internal[:nc-npr] = lcp.zint_tubo(omega, rhoc, rf, ri, 1.0)
        z_internal[nc-npr:] = lcp.zin(omega, rhopr, rpr, 1.0)
else:
        z_internal[:nc-npr] = lcp.zin(omega, rhoc, rf, 1.0)
        z_internal[nc-npr:] = lcp.zin(omega, rhopr, rpr, 1.0)
    
zin_matrix = np.diag(z_internal)

# External impedance
ze = lcp.compute_external_impedance_vectorized(omega, xc, yc, 1/rho_s, rf, rpr, npr)
print(ze) 

[[5.88028162e-05+0.00092624j 5.87681507e-05+0.00045434j
  5.87339250e-05+0.00050109j 5.86155748e-05+0.00041222j
  5.86157779e-05+0.00042578j]
 [5.87681507e-05+0.00045434j 5.87339250e-05+0.00092631j
  5.86993236e-05+0.00045441j 5.85814893e-05+0.00044114j
  5.85812863e-05+0.00042225j]
 [5.87339250e-05+0.00050109j 5.86993236e-05+0.00045441j
  5.86651611e-05+0.00092638j 5.85470295e-05+0.00043275j
  5.85472323e-05+0.00046042j]
 [5.86155748e-05+0.00041222j 5.85814893e-05+0.00044114j
  5.85470295e-05+0.00043275j 5.84296778e-05+0.00100832j
  5.84294757e-05+0.00045752j]
 [5.86157779e-05+0.00042578j 5.85812863e-05+0.00042225j
  5.85472323e-05+0.00046042j 5.84294757e-05+0.00045752j
  5.84296778e-05+0.00100832j]]
